In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# ============ 1. 读取数据 ============
print("读取数据...")

# 读取 preference 数据
df_pref = pd.read_csv('preference.csv')
print(f"✓ Preference 数据: {len(df_pref)} 行")

# 读取 ColBERT 检索结果
df_colbert = pd.read_csv('df_colbert_deduped.csv')
print(f"✓ ColBERT 结果: {len(df_colbert)} 行")

# 读取查询文本
df_queries = pd.read_csv('result_with_text.csv')
print(f"✓ 查询文本: {len(df_queries)} 行")

# ============ 2. 提取查询文本映射 ============
print("\n创建 qid -> query 映射...")

# 创建 qid 到 query_text 的映射
if 'query_text' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query_text'].first().to_dict()
    print(f"✓ 从 result_with_text.csv 提取了 {len(qid_to_query)} 个查询")
elif 'query' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query'].first().to_dict()
    print(f"✓ 从 result_with_text.csv 提取了 {len(qid_to_query)} 个查询")
else:
    print("✗ 找不到 query_text 或 query 列")
    print(f"可用列: {df_queries.columns.tolist()}")
    qid_to_query = {}

# 添加查询文本到 preference 数据
df_pref['query'] = df_pref['qid'].map(qid_to_query)

matched_count = df_pref['query'].notna().sum()
print(f"✓ 成功匹配 {matched_count}/{len(df_pref)} 个查询")

if matched_count < len(df_pref):
    missing_qids = df_pref[df_pref['query'].isna()]['qid'].tolist()
    print(f"⚠️ 缺失查询的 QID: {missing_qids[:5]}..." if len(missing_qids) > 5 else f"⚠️ 缺失查询的 QID: {missing_qids}")

# ============ 3. 为每个查询提取 Top-5 文档 ============
print("\n提取 Top-5 文档...")

top5_data = []

for qid in df_pref['qid'].unique():
    # 获取该查询的检索结果
    query_results = df_colbert[df_colbert['qid'] == qid].sort_values('score', ascending=False)
    
    # Top-5 文档
    top5 = query_results.head(5)
    
    if len(top5) == 0:
        continue
    
    # 格式化为字符串
    top5_texts = []
    for i, (idx, row) in enumerate(top5.iterrows()):
        text = str(row.get('passage_text', ''))
        if text and text != 'nan':
            top5_texts.append(f"[{i+1}] Score: {row['score']:.3f}\n    {text[:200]}...")
        else:
            top5_texts.append(f"[{i+1}] Score: {row['score']:.3f} | DocID: {row['docno']}")
    
    top5_summary = "\n\n".join(top5_texts)
    
    # 获取所有 Top-5 的 docno
    top5_docnos = ','.join(top5['docno'].astype(str).tolist())
    
    top5_data.append({
        'qid': qid,
        'top5_summary': top5_summary,
        'top5_docnos': top5_docnos,
        'top5_count': len(top5)
    })

df_top5 = pd.DataFrame(top5_data)
print(f"✓ 提取了 {len(df_top5)} 个查询的 Top-5 文档")

# ============ 4. 合并数据 ============
print("\n合并数据...")
df_merged = df_pref.merge(df_top5, on='qid', how='left')
print(f"✓ 合并后: {len(df_merged)} 行")

# ============ 5. 计算查询特征 ============
print("\n计算查询特征...")

# 5.1 查询长度
df_merged['query_length'] = df_merged['query'].fillna('').str.split().str.len()

# 5.2 查询类型
def classify_query_type(query):
    if pd.isna(query) or query == '':
        return 'unknown'
    q = str(query).lower()
    if any(w in q for w in ['what', 'who', 'when', 'where', 'why', 'how']):
        return 'factoid'
    elif any(w in q for w in ['best', 'top', 'recommend', 'good']):
        return 'recommendation'
    elif any(w in q for w in ['define', 'definition', 'meaning', 'mean']):
        return 'definition'
    elif '?' in q:
        return 'question'
    else:
        return 'other'

df_merged['query_type'] = df_merged['query'].apply(classify_query_type)

# 5.3 计算代表性得分
print("计算代表性得分...")

queries_valid = df_merged[df_merged['query'].notna() & (df_merged['query'] != '')]['query'].tolist()

if len(queries_valid) > 1:
    try:
        vectorizer = TfidfVectorizer(max_features=100, min_df=1)
        query_vectors = vectorizer.fit_transform(queries_valid)
        similarity_matrix = cosine_similarity(query_vectors)
        
        # 排除自己
        np.fill_diagonal(similarity_matrix, 0)
        
        # 平均相似度
        avg_similarity = similarity_matrix.mean(axis=1)
        
        # 添加回 dataframe
        valid_indices = df_merged[df_merged['query'].notna() & (df_merged['query'] != '')].index
        df_merged.loc[valid_indices, 'representativeness'] = avg_similarity
        print(f"✓ 计算了 {len(avg_similarity)} 个查询的代表性得分")
    except Exception as e:
        print(f"⚠️ 代表性计算失败: {e}")
        df_merged['representativeness'] = 0.5
else:
    print("⚠️ 查询数量不足，使用默认代表性得分")
    df_merged['representativeness'] = 0.5

# 填充缺失值
df_merged['representativeness'] = df_merged['representativeness'].fillna(0.5)

# 5.4 判断是否为极端案例
if df_merged['representativeness'].nunique() > 1:
    threshold_low = df_merged['representativeness'].quantile(0.25)
    threshold_high = df_merged['representativeness'].quantile(0.75)
    df_merged['is_extreme'] = (
        (df_merged['representativeness'] < threshold_low) |
        (df_merged['representativeness'] > threshold_high)
    )
else:
    df_merged['is_extreme'] = False

# 5.5 PRF benefit 强度分类
def classify_prf_strength(row):
    ratio = row.get('b_preference_ratio', 0.5)
    pref = row.get('preference', 'Neutral')
    
    if pref == 'Neutral':
        return 'Neutral'
    elif pref == 'PRF-Benefit':
        if ratio > 0.7:
            return 'Strong-Benefit'
        else:
            return 'Moderate-Benefit'
    elif pref == 'PRF-Hurt':
        if ratio < 0.3:
            return 'Strong-Hurt'
        else:
            return 'Moderate-Hurt'
    else:
        return 'Unknown'

df_merged['prf_strength'] = df_merged.apply(classify_prf_strength, axis=1)

# ============ 6. 计算选择得分 ============
print("\n计算 Few-Shot 选择得分...")

def calculate_selection_score(row):
    # 代表性（0.4权重）
    repr_score = row.get('representativeness', 0.5)
    
    # PRF 效果明显性（0.4权重）
    ratio = row.get('b_preference_ratio', 0.5)
    clarity_score = abs(ratio - 0.5)  # 越接近 0 或 1 越好
    
    # 查询长度适中（0.2权重）
    length = row.get('query_length', 0)
    if length > 0:
        median_length = df_merged[df_merged['query_length'] > 0]['query_length'].median()
        if median_length > 0:
            length_score = 1 - abs(length - median_length) / median_length
            length_score = max(0, min(1, length_score))
        else:
            length_score = 0.5
    else:
        length_score = 0
    
    return (repr_score * 0.4 + 
            clarity_score * 0.4 + 
            length_score * 0.2)

df_merged['selection_score'] = df_merged.apply(calculate_selection_score, axis=1)

# ============ 7. 筛选候选样本 ============
print("\n筛选候选样本...")

# 非极端样本
df_candidates = df_merged[~df_merged['is_extreme']].copy()

# 确保有查询文本
df_candidates = df_candidates[df_candidates['query'].notna() & (df_candidates['query'] != '')]

# 确保有 Top-5 文档
df_candidates = df_candidates[df_candidates['top5_count'] >= 3]

print(f"候选样本数: {len(df_candidates)}")
if len(df_candidates) > 0:
    print(f"  - PRF-Benefit: {(df_candidates['preference']=='PRF-Benefit').sum()}")
    print(f"  - PRF-Hurt: {(df_candidates['preference']=='PRF-Hurt').sum()}")
    print(f"  - Neutral: {(df_candidates['preference']=='Neutral').sum()}")

# ============ 8. 生成输出 CSV ============
print("\n生成输出文件...")

# 选择输出列
output_columns = [
    'qid',
    'query',
    'query_length',
    'query_type',
    'preference',
    'b_preference_ratio',
    'prf_strength',
    'representativeness',
    'is_extreme',
    'selection_score',
    'top5_summary',
    'top5_docnos'
]

# 按选择得分排序
df_output = df_merged[output_columns].copy().sort_values('selection_score', ascending=False)

# 保存完整数据
df_output.to_csv('few_shot_candidates_full.csv', index=False, encoding='utf-8')
print(f"✓ 完整数据已保存: few_shot_candidates_full.csv ({len(df_output)} 行)")

# 保存候选数据（非极端）
if len(df_candidates) > 0:
    df_candidates_output = df_candidates[output_columns].copy().sort_values('selection_score', ascending=False)
    df_candidates_output.to_csv('few_shot_candidates_filtered.csv', index=False, encoding='utf-8')
    print(f"✓ 候选数据已保存: few_shot_candidates_filtered.csv ({len(df_candidates_output)} 行)")

# ============ 9. 推荐 Top Few-Shot 样本 ============
print("\n" + "="*60)
print("推荐 Few-Shot 样本")
print("="*60)

if len(df_candidates) > 0:
    # 从不同类别中选择
    benefit_samples = df_candidates[
        df_candidates['preference'] == 'PRF-Benefit'
    ].nlargest(3, 'selection_score')
    
    hurt_samples = df_candidates[
        df_candidates['preference'] == 'PRF-Hurt'
    ].nlargest(3, 'selection_score')
    
    # 保存推荐样本
    recommended_samples = pd.concat([benefit_samples, hurt_samples])
    if len(recommended_samples) > 0:
        recommended_samples.to_csv('few_shot_recommended.csv', index=False, encoding='utf-8')
        print(f"✓ 推荐样本已保存: few_shot_recommended.csv ({len(recommended_samples)} 行)")
        
        # 打印摘要
        print("\n推荐的 Few-Shot 样本:")
        print("-" * 60)
        
        if len(benefit_samples) > 0:
            print(f"\n【PRF-Benefit】({len(benefit_samples)} 个)")
            for idx, (_, row) in enumerate(benefit_samples.iterrows(), 1):
                print(f"\n{idx}. Query: {row['query']}")
                print(f"   QID: {row['qid']}")
                print(f"   Length: {row['query_length']} words")
                print(f"   Type: {row['query_type']}")
                print(f"   PRF Ratio: {row['b_preference_ratio']:.3f}")
                print(f"   Strength: {row['prf_strength']}")
                print(f"   Selection Score: {row['selection_score']:.3f}")
        
        if len(hurt_samples) > 0:
            print(f"\n【PRF-Hurt】({len(hurt_samples)} 个)")
            for idx, (_, row) in enumerate(hurt_samples.iterrows(), 1):
                print(f"\n{idx}. Query: {row['query']}")
                print(f"   QID: {row['qid']}")
                print(f"   Length: {row['query_length']} words")
                print(f"   Type: {row['query_type']}")
                print(f"   PRF Ratio: {row['b_preference_ratio']:.3f}")
                print(f"   Strength: {row['prf_strength']}")
                print(f"   Selection Score: {row['selection_score']:.3f}")
else:
    print("⚠️ 没有找到合适的候选样本")

# ============ 10. 统计信息 ============
print("\n" + "="*60)
print("数据统计")
print("="*60)

print(f"\n总样本数: {len(df_merged)}")
print(f"有查询文本: {(df_merged['query'].notna() & (df_merged['query'] != '')).sum()}")
print(f"有 Top-5 文档: {df_merged['top5_count'].notna().sum()}")
print(f"候选样本数（非极端）: {len(df_candidates)}")

print("\nPreference 分布:")
print(df_merged['preference'].value_counts())

if (df_merged['query'] != '').any():
    print("\n查询类型分布:")
    print(df_merged['query_type'].value_counts())
    
    print("\n查询长度统计:")
    length_stats = df_merged[df_merged['query_length'] > 0]['query_length'].describe()
    print(length_stats)

print("\nPRF Strength 分布:")
print(df_merged['prf_strength'].value_counts())

print("\n" + "="*60)
print("✓ 完成！请查看生成的 CSV 文件")
print("="*60)

print("\n生成的文件:")
print("1. few_shot_candidates_full.csv - 所有样本（按 selection_score 排序）")
if len(df_candidates) > 0:
    print("2. few_shot_candidates_filtered.csv - 候选样本（排除极端案例）")
    print("3. few_shot_recommended.csv - 推荐的 6 个样本（3 Benefit + 3 Hurt）")

print("\n下一步:")
print("1. 打开 few_shot_recommended.csv 查看推荐样本")
print("2. 查看 top5_summary 列，确认文档内容是否合适")
print("3. 人工审核并最终选择 5 个样本")
print("4. 把选好的样本发给我，我帮你生成 Few-Shot Prompt 模板")

读取数据...
✓ Preference 数据: 43 行
✓ ColBERT 结果: 281534 行
✓ 查询文本: 387 行

创建 qid -> query 映射...
✓ 从 result_with_text.csv 提取了 43 个查询
✓ 成功匹配 43/43 个查询

提取 Top-5 文档...
✓ 提取了 43 个查询的 Top-5 文档

合并数据...
✓ 合并后: 43 行

计算查询特征...
计算代表性得分...
✓ 计算了 43 个查询的代表性得分

计算 Few-Shot 选择得分...

筛选候选样本...
候选样本数: 21
  - PRF-Benefit: 5
  - PRF-Hurt: 5
  - Neutral: 10

生成输出文件...
✓ 完整数据已保存: few_shot_candidates_full.csv (43 行)
✓ 候选数据已保存: few_shot_candidates_filtered.csv (21 行)

推荐 Few-Shot 样本
✓ 推荐样本已保存: few_shot_recommended.csv (6 行)

推荐的 Few-Shot 样本:
------------------------------------------------------------

【PRF-Benefit】(3 个)

1. Query: who is robert gray
   QID: 1037798
   Length: 4 words
   Type: factoid
   PRF Ratio: 0.930
   Strength: Strong-Benefit
   Selection Score: 0.348

2. Query: is cdg airport in main paris
   QID: 405717
   Length: 6 words
   Type: other
   PRF Ratio: 0.905
   Strength: Strong-Benefit
   Selection Score: 0.334

3. Query: lps laws definition
   QID: 443396
   Length: 3 words
   Type: defi

In [4]:
import pandas as pd

# ============ 配置 ============
print("请选择方案:")
print("A - 包含 Strong-Hurt (展示极端案例)")
print("B - 全用 Moderate (避免极端)")
print()

# 方案 A: 包含 Strong-Hurt
selected_qids_A = [
    131843,   # definition of a sigmet (Strong-Benefit)
    1037798,  # who is robert gray (Strong-Benefit)
    1110199,  # what is wifi vs bluetooth (Moderate-Benefit)
    855410,   # what is theraderm used for (Strong-Hurt - 极端)
    87452     # causes of military suicide (Moderate-Hurt)
]

# 方案 B: 全用 Moderate
selected_qids_B = [
    131843,   # definition of a sigmet (Strong-Benefit)
    1037798,  # who is robert gray (Strong-Benefit)
    1110199,  # what is wifi vs bluetooth (Moderate-Benefit)
    451602,   # medicare's definition of mechanical ventilation (Moderate-Hurt)
    87452     # causes of military suicide (Moderate-Hurt)
]

# 默认使用方案 A
selected_qids = selected_qids_A
plan_name = "A"

# 如果要用方案 B，修改这里
# selected_qids = selected_qids_B
# plan_name = "B"

print(f"使用方案 {plan_name}")
print(f"选中的 QID: {selected_qids}")
print()

# ============ 读取数据 ============
print("读取数据...")
df_colbert = pd.read_csv('df_colbert_deduped.csv')
df_queries = pd.read_csv('result_with_text.csv')

print(f"✓ ColBERT 结果: {len(df_colbert)} 行")
print(f"✓ 查询数据: {len(df_queries)} 行")

# 创建 qid -> query 映射
if 'query_text' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query_text'].first().to_dict()
elif 'query' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query'].first().to_dict()
else:
    print("⚠️ 找不到查询文本列")
    qid_to_query = {}

# ============ 提取 Top-5 文档 ============
print("\n" + "="*80)
print(f"提取方案 {plan_name} 的 5 个查询的 Top-5 文档")
print("="*80)

results = []

for qid in selected_qids:
    print(f"\n{'='*80}")
    print(f"QID: {qid}")
    
    # 获取查询文本
    query_text = qid_to_query.get(qid, f"[Query text not found for QID {qid}]")
    print(f"Query: {query_text}")
    print("-" * 80)
    
    # 获取该查询的检索结果
    query_results = df_colbert[df_colbert['qid'] == qid].sort_values('score', ascending=False)
    
    # Top-5
    top5 = query_results.head(5)
    
    if len(top5) == 0:
        print(f"⚠️ 没有找到该查询的检索结果")
        continue
    
    # 打印每个文档
    for rank, (idx, row) in enumerate(top5.iterrows(), 1):
        docno = row['docno']
        score = row['score']
        text = str(row.get('passage_text', ''))
        
        print(f"\n[{rank}] DocID: {docno} | Score: {score:.3f}")
        print(f"    {text[:300]}...")
        
        # 保存到结果
        results.append({
            'qid': qid,
            'query': query_text,
            'rank': rank,
            'docno': docno,
            'score': score,
            'passage_text': text
        })

# ============ 保存结果 ============
df_results = pd.DataFrame(results)
output_csv = f'selected_5_queries_plan_{plan_name}_top5_docs.csv'
df_results.to_csv(output_csv, index=False, encoding='utf-8')

print("\n" + "="*80)
print(f"✓ 完成！结果已保存到: {output_csv}")
print(f"总计: {len(df_results)} 个文档 (5 查询 × 5 文档)")
print("="*80)

# ============ 生成 Markdown 格式报告 ============
print("\n生成详细报告...")

markdown_report = f"""# Few-Shot 样本详细文档 (方案 {plan_name})

## 选中的 5 个查询及其 Top-5 文档

"""

for qid in selected_qids:
    query_text = qid_to_query.get(qid, "Unknown")
    query_docs = df_results[df_results['qid'] == qid]
    
    markdown_report += f"""
---

### QID: {qid}
**Query**: {query_text}

"""
    
    for idx, row in query_docs.iterrows():
        markdown_report += f"""
#### [{row['rank']}] Score: {row['score']:.3f} | DocID: {row['docno']}

{row['passage_text']}

"""

# 保存 Markdown 报告
output_md = f'selected_5_queries_plan_{plan_name}_detailed_report.md'
with open(output_md, 'w', encoding='utf-8') as f:
    f.write(markdown_report)

print(f"✓ 详细报告已保存到: {output_md}")

print("\n" + "="*80)
print("✓ 全部完成！")
print("="*80)
print("\n生成的文件:")
print(f"1. {output_csv} - CSV 格式")
print(f"2. {output_md} - Markdown 格式")
print("\n下一步:")
print("1. 打开 Markdown 文件查看完整内容")
print("2. 把文件发给 Claude")
print("3. Claude 会帮你生成最终的 Few-Shot Prompt 模板！")

请选择方案:
A - 包含 Strong-Hurt (展示极端案例)
B - 全用 Moderate (避免极端)

使用方案 A
选中的 QID: [131843, 1037798, 1110199, 855410, 87452]

读取数据...
✓ ColBERT 结果: 281534 行
✓ 查询数据: 387 行

提取方案 A 的 5 个查询的 Top-5 文档

QID: 131843
Query: definition of a sigmet
--------------------------------------------------------------------------------

[1] DocID: 8305152 | Score: 28.223
    Definition of SIGMET. plural. SIGMETs. also. sigmets. : a notice of significant hazardous weather conditions (such as extreme turbulence, icing, or poor visibility) in a large area that is provided to the pilot of an aircraft before takeoff — compare airmet....

[2] DocID: 8305153 | Score: 25.970
    SIGMET, or Significant Meteorological Information, is a weather advisory that contains meteorological information concerning the safety of all aircraft. There are two types of SIGMETs, convective and non-convective....

[3] DocID: 8305156 | Score: 25.511
    SIGMET, or Significant Meteorological Information, is a weather advisory that contains